# 实践 4.3  GWAS功效评估：样本量×MAF×效应大小

《统计遗传与智能育种学》第四章 · 电子资源 `04c_gwas_power.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：04c_gwas_power.ipynb。实践构造单个真实因果SNP，在多个样本量、MAF和每等位基因效应β下重复生成表型；基因型只中心化、不标准化，因此在Hardy–Weinberg平衡的模拟条件下，MAF会通过2p(1−p)进入标记方差。Notebook以线性回归统计经验功效，同时给出近似标记R²，并模拟完全无效SNP检查经验I类错误。该实践可直接比较低频＋小效应与较常见等位基因的检测难度，以及显著性阈值收紧后的功效损失。

三因素：n、MAF、per-allele effect  
输出：approx_marker_R2、经验Power、Type I error与功效数据表  
练习：固定n、MAF和每等位基因效应，将残差标准差从1.0改为1.5，并将approx_marker_R2分母中的残差方差由1.0改为2.25，解释标记解释方差比例和检测功效为何同时变化。

## 输入、完整代码、实际输出与解释

输入与可复现条件：单个因果SNP，n=120/300/700，MAF=0.05/0.20/0.40，每等位基因效应β=0.10/0.25/0.45；SEED=403；每个条件重复140次模拟。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=403
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 403


In [2]:
from scipy.stats import t as tdist
def assoc_p(x,y):
 if np.ptp(x)==0: return 1.0
 X=np.column_stack([np.ones(len(x)),x]); b=np.linalg.lstsq(X,y,rcond=None)[0]; r=y-X@b; df=len(y)-2; s2=(r@r)/df; se=np.sqrt(s2*np.linalg.inv(X.T@X)[1,1]); return float(2*tdist.sf(abs(b[1]/se),df))
def power(n,maf,effect,alpha,reps=140,resid_sd=1.0):
 hit=0
 for _ in range(reps):
  x=rng.binomial(2,maf,n); xc=x-x.mean(); y=effect*xc+rng.normal(0,resid_sd,n); hit+=assoc_p(x,y)<alpha
 return hit/reps
rows=[]
for n in [120,300,700]:
 for maf in [.05,.2,.4]:
  for e in [.1,.25,.45]:
   marker_var=2*maf*(1-maf)*e**2; r2=marker_var/(marker_var+1.0)
   rows.append((n,maf,e,r2,power(n,maf,e,.05),power(n,maf,e,5e-4)))
print(pd.DataFrame(rows,columns=['n','MAF','per_allele_effect','approx_marker_R2','power_a05','power_a5e-4']).round(3).to_string(index=False))

# 完全无效SNP的经验I类错误
vals=[]
for _ in range(500):
 x=rng.binomial(2,.3,300); y=rng.normal(size=300); vals.append(assoc_p(x,y))
print('empirical Type-I error at 0.05:',round(float(np.mean(np.array(vals)<.05)),3))


  n  MAF  per_allele_effect  approx_marker_R2  power_a05  power_a5e-4
120 0.05               0.10             0.001      0.079        0.000
120 0.05               0.25             0.006      0.107        0.007
120 0.05               0.45             0.019      0.343        0.029
120 0.20               0.10             0.003      0.086        0.000
120 0.20               0.25             0.020      0.379        0.021
120 0.20               0.45             0.061      0.764        0.221
120 0.40               0.10             0.005      0.129        0.000
120 0.40               0.25             0.029      0.429        0.071
120 0.40               0.45             0.089      0.914        0.407
300 0.05               0.10             0.001      0.064        0.000
300 0.05               0.25             0.006      0.279        0.021
300 0.05               0.45             0.019      0.679        0.114
300 0.20               0.10             0.003      0.171        0.000
300 0.20            

结果解释：在基因型只中心化、不标准化的设定下，样本量、MAF和效应共同决定标记方差及经验功效；更严格阈值5×10⁻⁴显著降低弱效应检测率。完全无效SNP在α=0.05下经验I类错误0.046，该设定下未见明显的I类错误膨胀。表中单次蒙特卡洛power不应读成现实项目的固定保证。

可修改练习：把重复次数提高到1000，并为power计算Monte Carlo标准误；增加MAF=0.01，讨论低频变异在同一每等位基因效应下为什么更难检测。